# Практикум 01 · Чтение изображения и первый результат

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 01. Первое изображение: как компьютер видит картинку

**Тема Yahboom:** 9.1 Getting started with Open Source CV. **Время:** 45–60 минут плюс самостоятельное задание.

## Цель и место выполнения

После занятия вы сможете открыть изображение программой, узнать его размеры, прочитать цвет одного пикселя и сохранить результат обработки. Эти действия нужны перед работой с камерой ROSMASTER: видеопоток состоит из отдельных кадров, к которым применяются такие же операции.

Выполняйте урок на своём ноутбуке или в терминале компьютера ROSMASTER из **корня папки курса**. Подготовьте окружение по [START_HERE](../START_HERE.md). Все команды используют `python3`; если ваша система использует другое имя Python, следуйте стартовой инструкции. Робот во время урока неподвижен. Программа работает с файлом и не требует камеры, ROS или открывающегося графического окна.

## Четыре понятия

**Пиксель** — маленький элемент изображения. У нашей учебной картинки 800 столбцов и 600 строк: всего 480 000 пикселей. Каждый пиксель цветного изображения содержит три числовых значения.

**BGR** — порядок цветовых каналов при обычном чтении в OpenCV: синий, зелёный, красный. Например, `(0, 0, 255)` означает яркий красный. В этих упражнениях значения каналов имеют тип `uint8`: целые числа от 0 до 255. Большое значение красного канала ещё не означает красный цвет: у белого пикселя велики все три канала.

**Координаты** начинаются в левом верхнем углу. `x` растёт вправо, `y` — вниз; первая точка имеет координаты `(0, 0)`. В массиве Python сначала указывается строка, поэтому запись имеет вид `image[y, x]`. Размер массива записывается в порядке «высота, ширина, каналы».

**Оттенки серого** представляют яркость одним числом на пиксель. `cvtColor` использует взвешенное сочетание цветовых каналов. Полученный массив двумерен. Изображение сохраняет формы и светлые области, но часть информации о различии цветов теряется.

## Запуск на нашем ROSMASTER

Это готовый маршрут для учебного стенда: **IP `192.168.1.185`, пользователь `jetson`, папка курса `/home/jetson/rosmaster-cv-course`**. Ноутбук должен быть подключён к той же локальной сети. Введите действующий пароль учётной записи `jetson` или используйте настроенный SSH-ключ; секреты в отчёт не записывают. На этом стенде проверены Python **3.10.12** и OpenCV **4.10.0**. Первый урок выполняется в обычной системе Jetson: запускать Docker, ROS или камеру не требуется.

В примерах `ivanov` — личное имя папки. **До запуска замените его во всех командах на свою уникальную фамилию латиницей или номер команды**. Так студенты сохраняют результаты отдельно. Повторный запуск с тем же `--out` перезаписывает файлы этого опыта.

### 1. Подключитесь с ноутбука

Откройте **PowerShell на Windows** либо терминал Linux/macOS и выполните:

```bash
ssh jetson@192.168.1.185
```

Введите пароль по запросу: символы при вводе не отображаются. После входа команды выполняются **на роботе**. Перейдите в подготовленную папку и проверьте окружение:

```bash
cd /home/jetson/rosmaster-cv-course
pwd
python3 -c "import sys, cv2, numpy; print('Python:', sys.version.split()[0]); print('OpenCV:', cv2.__version__); print('NumPy:', numpy.__version__)"
ls data/scene.png
mkdir -p outputs/ivanov/01 student_photos/ivanov
```

`pwd` должен показать `/home/jetson/rosmaster-cv-course`. Если папка отсутствует, создайте её командой `git clone https://github.com/tolenserik-source/rosmaster-cv-course.git /home/jetson/rosmaster-cv-course`, затем повторите `cd`. Если каталог уже существует, не клонируйте поверх него. Для отсутствующего комплектного `data/scene.png` в этой Git-копии выполните `git restore --source=HEAD -- data/scene.png`; свои фотографии этой командой не восстанавливают.

При ошибке импорта проверьте `/usr/bin/python3 -c "import cv2, numpy; print(cv2.__version__, numpy.__version__)"`. Если проверка прошла, используйте `/usr/bin/python3` во всех запусках. Если не прошла, подготовьте отдельное окружение ноутбука по [инструкции](../docs/SETUP_ROBOT.md#6-установка-на-личном-ноутбуке). Заводские библиотеки и общие данные не переустанавливайте ради первого упражнения.

### 2. Выполните три запуска на роботе

Перед измерением цветных точек предположите, какой канал окажется наибольшим. Затем выполните в **SSH-терминале**:

```bash
python3 -m rosmaster_cv.lesson01_intro --input data/scene.png --out outputs/ivanov/01/baseline
python3 -m rosmaster_cv.lesson01_intro --input data/scene.png --pixel 160 200 --out outputs/ivanov/01/red
python3 -m rosmaster_cv.lesson01_intro --input data/scene.png --pixel 650 420 --out outputs/ivanov/01/blue
cat outputs/ivanov/01/baseline/report.json
ls outputs/ivanov/01/baseline
```

При успешном запуске создаются шесть файлов на опыт: `original.png`, `grayscale.png`, `channel_blue.png`, `channel_green.png`, `channel_red.png`, `report.json`. У исходной сцены размер **800 × 600**, три канала и тип `uint8`.

| Опыт | `pixel_xy` — x, y | `pixel_bgr` — B, G, R | `pixel_gray` |
|---|---|---|---|
| Центр (`baseline`) | `[400, 300]` | `[64, 59, 54]` | `58` |
| Красный круг (`red`) | `[160, 200]` | `[31, 44, 225]` | `97` |
| Синий прямоугольник (`blue`) | `[650, 420]` | `[225, 96, 31]` | `91` |

Эти значения проверены для комплектного `data/scene.png`. Сравните их со своими JSON. Окно с картинкой на роботе не открывается — это ожидаемое поведение.

### 3. Скачайте результаты и откройте их на ноутбуке

Оставьте SSH-терминал открытым. Откройте **второе окно PowerShell на своём ноутбуке**, выполните:

```powershell
New-Item -ItemType Directory -Force "$HOME\Downloads\rosmaster-results\ivanov" | Out-Null
Set-Location "$HOME\Downloads\rosmaster-results\ivanov"
scp -r jetson@192.168.1.185:/home/jetson/rosmaster-cv-course/outputs/ivanov/01 .
Start-Process .\01\baseline\original.png
Start-Process .\01\baseline\grayscale.png
Start-Process .\01\red\channel_red.png
Get-Content .\01\red\report.json
```

`scp` запросит пароль робота. Результаты появятся на ноутбуке в `Downloads\rosmaster-results\ivanov\01`. Откройте эту папку через Проводник, чтобы сравнить остальные каналы и отчёты. Команды `Start-Process` и `Get-Content` относятся к PowerShell; их не вводят в SSH-сеанс.

На **Linux/macOS** во втором локальном терминале используйте:

```bash
mkdir -p ~/Downloads/rosmaster-results/ivanov
cd ~/Downloads/rosmaster-results/ivanov
scp -r jetson@192.168.1.185:/home/jetson/rosmaster-cv-course/outputs/ivanov/01 .
```

Откройте скачанную папку `01` в файловом менеджере, изображения — двойным щелчком, JSON — текстовым редактором.

### 4. Обработайте собственную фотографию

Сделайте фотографию стола и сохраните её как `my_table.jpg` в локальную папку `Downloads/rosmaster-results/ivanov`. В том же **локальном PowerShell** отправьте файл на робота:

```powershell
scp .\my_table.jpg jetson@192.168.1.185:/home/jetson/rosmaster-cv-course/student_photos/ivanov/table.jpg
```

На Linux/macOS в этой команде замените `.\my_table.jpg` на `./my_table.jpg`. Затем в **SSH-терминале робота** выполните:

```bash
cd /home/jetson/rosmaster-cv-course
python3 -m rosmaster_cv.lesson01_intro --input student_photos/ivanov/table.jpg --out outputs/ivanov/01/own
cat outputs/ivanov/01/own/report.json
```

Запишите реальные размеры, центральный пиксель и то, что видно на фотографии. Выполните также три измерения из раздела «Самостоятельное изменение» ниже, сохраняя их в отдельные папки внутри `outputs/ivanov/01/`, например `green`, `light`, `dark`. Во втором локальном терминале повторите команду `scp -r` из шага 3: она скопирует новые и обновит существующие результаты в той же папке `01`. Откройте `01/own/original.png` и `01/own/grayscale.png`.

### 5. Оформите и сдайте отчёт

Прочитайте [пример отчёта первого урока](../templates/REPORT_01_EXAMPLE.md): в нём заполнена часть на учебной сцене и показано, как добавить свои данные. После последнего скачивания результатов получите пустой шаблон **на ноутбук**. В локальном PowerShell, находясь в `Downloads\rosmaster-results\ivanov`, выполните **один раз**:

```powershell
scp jetson@192.168.1.185:/home/jetson/rosmaster-cv-course/templates/REPORT.md .\01\REPORT.md
notepad .\01\REPORT.md
```

На Linux/macOS укажите `./01/REPORT.md` и откройте его текстовым редактором. Повторное скачивание шаблона заменяет заполненный файл, поэтому дальше редактируйте свою локальную копию.

Заполните ФИО, группу, дату, версии, команды, результаты и ответы. Приложите собственную фотографию, её серую версию и три самостоятельных измерения. Сдайте папку `01` целиком или ZIP этой папки: в ней должны быть `REPORT.md` и папки опытов с PNG/JSON. Сохраните её на ноутбуке до окончания занятия. Чтобы выйти из SSH, выполните `exit` в терминале робота.

**Если связь пропала:** сохранённые на ноутбуке файлы остаются доступны. При самопроизвольном выключении или перезагрузке выполните [проверку питания и сети](../docs/TROUBLESHOOTING.md#подключение-и-перезагрузка). Затем подключитесь заново, выполните `cd /home/jetson/rosmaster-cv-course` и проверьте результаты через `ls`. Незавершённый опыт повторите в новой папке `--out`, например `outputs/ivanov/01/baseline_retry`.

## Практика в собственной копии курса: семь шагов

Если вы уже выполнили маршрут на нашем ROSMASTER, переходите к самостоятельному изменению ниже. Следующие семь шагов — альтернативный маршрут для личной копии курса на ноутбуке. На общем роботе используйте личные папки из предыдущего раздела.

1. Откройте терминал в папке курса. Посмотрите доступные параметры:

   ```bash
   python3 -m rosmaster_cv.lesson01_intro --help
   ```

2. Создайте оригинальные учебные данные. Генератор записывает свои файлы в `data/` внутри курса. Ваши фотографии сохраняйте под другими именами.

   ```bash
   python3 tools/generate_data.py
   ```

3. Выполните первый запуск:

   ```bash
   python3 -m rosmaster_cv.lesson01_intro --input data/scene.png --out outputs/01
   ```

4. Откройте `outputs/01/report.json` текстовым редактором. Найдите ширину, высоту, число каналов и значения центрального пикселя. Откройте `original.png` и `grayscale.png` обычным просмотрщиком изображений. При работе через SSH перенесите папку результатов на ноутбук способом из стартовой инструкции.

5. Измерьте пиксель внутри красного круга:

   ```bash
   python3 -m rosmaster_cv.lesson01_intro --pixel 160 200 --out outputs/01_red
   ```

   Сначала предположите, какой канал будет наибольшим. Затем проверьте `pixel_bgr` в новом отчёте. Первая цифра в команде — `x`, вторая — `y`.

6. Исследуйте синий прямоугольник:

   ```bash
   python3 -m rosmaster_cv.lesson01_intro --pixel 650 420 --out outputs/01_blue
   ```

   Откройте три файла каналов. Они выглядят серыми: яркость каждого файла показывает величину одного канала исходного изображения. Сопоставьте светлые участки с цветными объектами.

7. Создайте папку `student_photos/`, положите туда собственную фотографию стола `my_table.jpg` и обработайте её:

   ```bash
   python3 -m rosmaster_cv.lesson01_intro --input student_photos/my_table.jpg --out outputs/01_own
   ```

   Запишите размеры и объясните, что находится в центре фотографии. Файл фотографии вы создаёте самостоятельно; в комплекте его нет.

## Ожидаемые результаты

Каждый запуск создаёт `original.png`, `grayscale.png`, `channel_blue.png`, `channel_green.png`, `channel_red.png` и `report.json`. Для стандартной сцены отчёт содержит `width: 800`, `height: 600`, `channels: 3`. Уменьшения изображения этот урок не выполняет.

## Самостоятельное изменение

Выберите ещё три точки: зелёный объект, светлый участок шкалы яркости и тёмный фон. Для каждой заранее запишите ожидаемый наибольший канал и примерную яркость. Запустите программу с собственными `--pixel` и отдельными `--out`. Составьте таблицу «точка — прогноз — измерение — объяснение». Найдите пример, где высокое значение красного канала присутствует у некрасного объекта.

## Три вопроса

1. Почему запись `image[200, 160]` соответствует точке `x=160, y=200`?
2. Чем различаются три значения BGR и одно значение серого пикселя?
3. Можно ли восстановить исходные цвета только по `grayscale.png`? Объясните причину.

## Что сдать и как исправить ошибки

Сдайте отчёты двух цветных точек, исходную и серую версию своей фотографии, таблицу трёх самостоятельных измерений и ответы. Укажите команду запуска и версию OpenCV. Используйте [пустой шаблон](../templates/REPORT.md) и [пример первого урока](../templates/REPORT_01_EXAMPLE.md). Одного скриншота терминала недостаточно для проверки ваших выводов.

**«Не найдено изображение».** Проверьте рабочую папку и имя после `--input`; выполните генератор, если нет `scene.png`. **«Пиксель вне изображения».** Для картинки шириной 800 последний `x` равен 799; координаты из учебной сцены не обязаны подходить к вашей фотографии.

## Источники и следующий шаг

Раздел производителя: [Yahboom 9.1](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8142/). Объяснения, код, данные и задания этого курса подготовлены отдельно. Чтение видеокадров продолжится в уроках о камере и ROS.

Перед следующим уроком создайте [личную копию курса](../docs/SETUP_ROBOT.md#0-подключитесь-и-создайте-личную-копию), чтобы свободно изменять код и сохранять отдельные результаты. Далее: [02 — Геометрические преобразования](../lessons/02_geometry.md).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


На общем ROSMASTER задайте своё имя папки в `STUDENT`. Ядро Jupyter должно работать на роботе: локальное ядро запускает код на ноутбуке. SSH и PowerShell из главы выполняются в терминалах. Учебные данные уже входят в курс.


In [ ]:
STUDENT = "ivanov"  # Замените на свою фамилию латиницей или номер команды.
if not STUDENT or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in STUDENT):
    raise ValueError("STUDENT: используйте строчные латинские буквы, цифры, _ или -.")
OUT = "outputs/" + STUDENT + "/01"


In [ ]:
run_module("lesson01_intro", "--input", "data/scene.png", "--out", OUT + "/baseline")
run_module("lesson01_intro", "--input", "data/scene.png", "--pixel", "160", "200", "--out", OUT + "/red")
run_module("lesson01_intro", "--input", "data/scene.png", "--pixel", "650", "420", "--out", OUT + "/blue")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results(OUT + "/baseline", limit=5)
print((ROOT / OUT / "red" / "report.json").read_text(encoding="utf-8"))


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
